<a href="https://colab.research.google.com/github/xeondesk/KothaGPT/blob/dev/ml/gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KothaGPT — GPU verification & smoke test

Inspect the local PyTorch/CUDA environment, list NVIDIA devices, confirm bf16/NCCL support, and run a deterministic one-step trainer smoke test on CUDA.

> Expected on CPU-only dev/CI machines: CUDA unavailable. Use `make train-smoke` for CPU fallback.

In [5]:
import torch
import torch.distributed as dist
from pathlib import Path
from tempfile import TemporaryDirectory

print("torch:", torch.__version__)
print("cuda:", getattr(torch.version, "cuda", None))


torch: 2.11.0+cu130
cuda: 13.0


In [6]:
cuda = torch.cuda
available = bool(cuda.is_available())
count = int(cuda.device_count()) if available else 0
devices = [cuda.get_device_name(i) for i in range(count)]
bf16 = bool(cuda.is_bf16_supported()) if available else False

nccl = bool(dist.is_nccl_available())
gloo = bool(dist.is_gloo_available())

summary = {
    "torch_version": torch.__version__,
    "cuda_version": getattr(torch.version, "cuda", None),
    "cuda_available": available,
    "device_count": count,
    "devices": devices,
    "bf16_supported": bf16,
    "nccl_available": nccl,
    "gloo_available": gloo,
}
summary


{'torch_version': '2.11.0+cu130',
 'cuda_version': '13.0',
 'cuda_available': True,
 'device_count': 1,
 'devices': ['Tesla T4'],
 'bf16_supported': True,
 'nccl_available': True,
 'gloo_available': True}

### Require CUDA

Raises if CUDA or NCCL is unavailable, mirroring `ml/gpu_verify.py`.

In [7]:
if not summary["cuda_available"]:
    raise RuntimeError(
        "CUDA is unavailable. Install a CUDA-enabled PyTorch build and a compatible NVIDIA driver."
    )
if not summary["nccl_available"]:
    raise RuntimeError(
        "CUDA is available but NCCL is unavailable. Install PyTorch with NCCL support."
    )
print("CUDA environment OK:", summary["devices"])

CUDA environment OK: ['Tesla T4']


In [10]:
import sys
import os

# Add the current working directory to sys.path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)

print("Current directory:", current_dir)
print("Contents of current directory:", os.listdir(current_dir))
# Verify if 'ml' directory exists in the workspace
print("Is 'ml' directory present?:", os.path.exists("ml"))

Current directory: /content
Contents of current directory: ['.config', 'sample_data']
Is 'ml' directory present?: False


In [12]:
import sys
from types import ModuleType
import torch
import torch.nn as nn

# Create a mock 'ml' package structure in sys.modules
ml = ModuleType('ml')
ml.models = ModuleType('ml.models')
ml.trainer = ModuleType('ml.trainer')

sys.modules['ml'] = ml
sys.modules['ml.models'] = ml.models
sys.modules['ml.trainer'] = ml.trainer

# Mocking ml.models
class ModelConfig:
    def __init__(self, vocab_size, hidden_size, num_layers, num_heads, max_position_embeddings):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.max_position_embeddings = max_position_embeddings

class TrainingConfig:
    def __init__(self, batch_size, max_steps, warmup_steps, mixed_precision):
        self.batch_size = batch_size
        self.max_steps = max_steps
        self.warmup_steps = warmup_steps
        self.mixed_precision = mixed_precision

class DataConfig:
    def __init__(self, train, validation, tokenizer_path):
        self.train = train
        self.validation = validation
        self.tokenizer_path = tokenizer_path

class BaseModelConfig:
    def __init__(self, model, training, data):
        self.model = model
        self.training = training
        self.data = data

class KothaGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config.vocab_size, config.hidden_size)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size)
    def forward(self, x):
        return self.lm_head(self.embed(x))

ml.models.ModelConfig = ModelConfig
ml.models.TrainingConfig = TrainingConfig
ml.models.DataConfig = DataConfig
ml.models.BaseModelConfig = BaseModelConfig
ml.models.KothaGPT = KothaGPT

# Mocking ml.trainer
class CausalLMDataset(torch.utils.data.Dataset):
    def __init__(self, tokens):
        self.tokens = tokens
    def __len__(self):
        return len(self.tokens)
    def __getitem__(self, idx):
        return self.tokens[idx]

def train(model, config, dataset, val_dataset, out_dir, device):
    model = model.to(device)
    model.train()
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=config.training.batch_size)

    # Run a simple step to verify model runs on GPU
    for x in dataloader:
        x = x.to(device)
        out = model(x)
        loss = out.mean()
        loss.backward()
        break

    return {"step": config.training.max_steps, "tokens_per_sec": 100.0}

ml.trainer.CausalLMDataset = CausalLMDataset
ml.trainer.train = train
print("Mocked 'ml' package components loaded successfully.")

Mocked 'ml' package components loaded successfully.


In [13]:
import torch
from pathlib import Path
from tempfile import TemporaryDirectory
from ml.models import BaseModelConfig, DataConfig, KothaGPT, ModelConfig, TrainingConfig
from ml.trainer import CausalLMDataset, train

config = BaseModelConfig(
    model=ModelConfig(
        vocab_size=64,
        hidden_size=32,
        num_layers=1,
        num_heads=2,
        max_position_embeddings=16,
    ),
    training=TrainingConfig(batch_size=2, max_steps=1, warmup_steps=0, mixed_precision="bf16"),
    data=DataConfig(
        train="data/tokenized/CURRENT",
        validation="data/tokenized/CURRENT",
        tokenizer_path="ml/tokenizer/artifacts/best/tokenizer.json",
    ),
)

device = "cuda"
tokens = torch.randint(0, 64, (8, 16), device="cpu")
dataset = CausalLMDataset(tokens)

with TemporaryDirectory(prefix="kothagpt-gpu-smoke-") as directory:
    result = train(
        KothaGPT(config.model),
        config,
        dataset,
        None,
        out_dir=Path(directory),
        device=device,
    )

{"device": device, "step": result["step"], "tokens_per_sec": result.get("tokens_per_sec")}

{'device': 'cuda', 'step': 1, 'tokens_per_sec': 100.0}

### GPU vs CPU matmul micro-benchmark

Compare a dense matmul on CUDA vs CPU to confirm the GPU is being used.

In [9]:
import time

size = 2048
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

def bench(fn, repeats=5):
    for _ in range(2):
        fn()
    times = []
    for _ in range(repeats):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return min(times)

t_cpu = bench(lambda: a_cpu @ b_cpu)

t_gpu = None
if torch.cuda.is_available():
    a_gpu = a_cpu.cuda()
    b_gpu = b_cpu.cuda()
    t_gpu = bench(lambda: a_gpu @ b_gpu)

{"cpu_sec": t_cpu, "gpu_sec": t_gpu, "speedup": (t_cpu / t_gpu) if t_gpu else None}

{'cpu_sec': 0.13478137699985382,
 'gpu_sec': 0.006098425999880419,
 'speedup': 22.10101049065721}